<a href="https://colab.research.google.com/github/mirian2004/AI-AI-/blob/Day20/Day20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**[사전준비] DB 설정**

In [19]:
!pip install -U -q google-genai

import sqlite3
import os
from google import genai
from google.genai import types
from google.colab import userdata

# 1. 설정 및 스키마 정의
os.environ["GEMINI_API_KEY"] = userdata.get('gemini_api_key')
DB_PATH = "online_shopping.db"

client = genai.Client()

DB_SCHEMA_TEXT = """
1. Table: customers (고객 정보)
   - Columns: customer_id, name, age, gender, membership(일반/VIP/VVIP), join_date

2. Table: products (상품 정보)
   - Columns: product_id, product_name, category, price, stock

3. Table: orders (주문 내역)
   - Columns: order_id, customer_id, product_id, order_date, quantity, total_amount, is_returned(0:정상, 1:반품)
"""

print("설정 완료!")




설정 완료!


**Text-to-SQL**

In [20]:
# 2. LLM : SQL 생성(Text-to-SQL)
def generate_sql(user_question):
  """
  사용자 질문을 SQL 쿼리로 변환하는 함수
  """

  system_prompt = f"""
  당신은 온라인 쇼핑몰의 데이터 분석가입니다.
  아래 제공된 데이터베이스 스키마 정보를 바탕으로 올바른 SQL Query를 작성하세요.

  [DB Schema Info]
  {DB_SCHEMA_TEXT}

  [규칙]
  1. 오직 실행 가능한 SQL Query문(SQLite 문법)만 출력하세요.
  2. 마크다운 기호(```sql)나 부가 설명을 절대 포함하지 마세요.
  """

  response = client.models.generate_content(
      model = 'gemini-flash-latest',
      contents = user_question,
      config = types.GenerateContentConfig(
          system_instruction = system_prompt,
          temperature = 0
      )
  )

  sql_query = response.text
  sql_query = sql_query.replace("```sql", "").replace("```", "")

  return sql_query

user_question = "카테고리별 매출 현황 알려줘"
print(generate_sql(user_question=user_question))



SELECT p.category, SUM(o.total_amount) AS total_sales, SUM(o.quantity) AS total_quantity, COUNT(o.order_id) AS total_orders
FROM products p
JOIN orders o ON p.product_id = o.product_id
WHERE o.is_returned = 0
GROUP BY p.category
ORDER BY total_sales DESC


**SQL 실행**

In [21]:
# 3. SQL 실행 (실제 DB 접속)
def execute_query(sql_query):
    """
    LLM이 생성한 쿼리를 실제 DB에서 실행하는 함수
    """
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    try:
        cursor.execute(sql_query)

        columns = [desc[0] for desc in cursor.description]

        results = cursor.fetchall()

        conn.close()
        return columns, results
    except Exception as e:
        conn.close()
        return None, f"SQL 실행 에러: {e}"

# 테스트
user_question = "카테고리별 매출 현황 알려줘"
generated_sql = generate_sql(user_question=user_question)
execute_query(sql_query=generated_sql)


(['category', 'total_sales'],
 [('전자기기', 19411000),
  ('명품', 6920000),
  ('가전제품', 2670000),
  ('게임', 1550000),
  ('패션/의류', 1263000),
  ('가구', 597000),
  ('생활용품', 490000)])

**답변 생성**

In [22]:
# 4. LLM: 답변 생성
def generate_explanation(user_question, results):
    prompt = f"""
    사용자 질문: "{user_question}"
    DB 조회 결과: {results}

    위 데이터를 바탕으로 사용자에게 친절하게 답변해주세요.
    결과가 비어있다면([]): "해당하는 내역이 없습니다"라고 답하세요.
    """

    response = client.models.generate_content(
        model= 'gemini-flash-latest',
        contents= prompt,
        config=types.GenerateContentConfig(
            temperature= 0.7
        )
    )

    return response.text

# 전체 파이프라인 테스트
user_question = "카테고리별 매출 현황 알려줘"
generated_sql = generate_sql(user_question=user_question)
cols, rows = execute_query(sql_query=generated_sql)
answer = generate_explanation(user_question=user_question, results=rows)
print(answer)


안녕하세요! 요청하신 카테고리별 매출 현황을 안내해 드립니다.

* **전자기기:** 19,411,000원
* **명품:** 6,920,000원
* **가전제품:** 2,670,000원
* **게임:** 1,550,000원
* **패션/의류:** 1,263,000원
* **가구:** 597,000원
* **생활용품:** 490,000원

**총 매출액:** 32,901,000원

추가로 궁금하신 사항이 있다면 언제든지 말씀해 주세요!


In [23]:
# 5. 전체 파이프라인 통합 함수
import time

def ask_database(question):
    generated_sql = generate_sql(user_question=question)
    time.sleep(15)  # SQL 생성 후 잠깐 대기
    cols, rows = execute_query(sql_query=generated_sql)
    answer = generate_explanation(user_question=question, results=rows)
    print(answer)
    time.sleep(15)  # 다음 질문 넘어가기 전 대기

# 테스트
ask_database("가장 많이 구매한 고객 3명은 누구야?")

가장 구매 금액이 높은 상위 고객 3명은 다음과 같습니다.

1. **강수민** 님: 9,800,000원
2. **한예린** 님: 7,070,000원
3. **김민준** 님: 5,308,000원

추가로 궁금한 점이 있으시면 언제든 말씀해 주세요!


**실제 적용**

In [24]:
# 6. 의심 고객 찾기
ask_database("반품(is_returned=1)을 2번 이상 한 고객은 누구야?")

ask_database("박지훈이 반품한 상품들의 이름과 가격을 알려줘")

ask_database("VVIP 고객들의 총 구매 금액은 얼마야?")


반품을 2번 이상 한 고객은 **박지훈** (고객 ID: 3) 님입니다.
박지훈님이 반품하신 상품의 이름과 가격 정보입니다.

* **삼성 갤럭시 S24**: 1,100,000원
* **다이슨 무선청소기**: 890,000원
* **닌텐도 스위치**: 450,000원
* **나이키 에어맥스**: 189,000원

추가로 궁금한 점이 있으시면 언제든 문의해 주세요!
VVIP 고객들의 총 구매 금액은 **16,870,000원**입니다.
